# Aetherius — Phi-3 Medium LoRA Training (HuggingFace · L40S)

Fine-tunes `microsoft/Phi-3-medium-4k-instruct` on the Aetherius corpus using full-precision BF16 LoRA.

**Platform**: HuggingFace JupyterLab — **Nvidia 1×L40S** (48 GB VRAM, $1.80/hr)

**Before running:**
1. In the HF JupyterLab UI, set `HF_TOKEN` as an environment variable (Secrets tab)
2. Upload `aetherius_corpus.jsonl` as a private HF dataset at `<your-namespace>/aetherius-corpus`
3. Update `CORPUS_REPO` and `OUTPUT_MODEL_ID` in Cell 2 if your namespace differs

**Output**: LoRA adapter pushed to `KingOfThoughtFleuren/phi3-aetherius` on HuggingFace Hub.

---
### Why no QLoRA on the L40S?
The L40S has 48 GB VRAM. Phi-3-medium (14B params) in BF16 uses ~28 GB, leaving 20 GB for LoRA
activations and optimizer state. Full BF16 trains ~20% faster than 4-bit and converges better.

### Tier strategy
- **Tier 1** (identity/character): instruction pairs × 4 → shapes personality
- **Tier 2** (frameworks): instruction pairs × 2 → shapes reasoning style
- **Tier 3** (knowledge): raw continuation × 1 → knowledge absorption

In [ ]:
# Cell 1 — Install training dependencies (run once per session)
!pip install -q \
    'transformers>=4.40.0' \
    'peft>=0.10.0' \
    'trl>=0.8.0' \
    'accelerate>=0.27.0' \
    'datasets>=2.18.0' \
    'huggingface-hub>=0.22.0'

import torch
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU  : {props.name}')
    print(f'VRAM : {props.total_memory / 1e9:.1f} GB')
    if props.total_memory < 40e9:
        print('WARNING: less than 40 GB VRAM detected — switch to the L40S instance.')

In [ ]:
# Cell 2 — Auth & configuration
import os
from huggingface_hub import login

# ── Token ─────────────────────────────────────────────────────────────────────
# Recommended: set HF_TOKEN in the JupyterLab Secrets/Env tab
# Fallback: paste your token below (never commit to git)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
# HF_TOKEN = 'hf_xxx'

if not HF_TOKEN:
    raise ValueError(
        'HF_TOKEN not found.\n'
        'Set it as a JupyterLab environment variable, or paste it above.'
    )

login(token=HF_TOKEN, add_to_git_credential=False)

# ── Model & data ──────────────────────────────────────────────────────────────
BASE_MODEL_ID   = 'microsoft/Phi-3-medium-4k-instruct'
OUTPUT_MODEL_ID = 'KingOfThoughtFleuren/phi3-aetherius'
CORPUS_REPO     = 'jonfleuren/aetherius-corpus'
OUTPUT_DIR      = '/tmp/phi3_aetherius_lora'

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_RANK       = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

# ── Training (L40S-optimised) ──────────────────────────────────────────────────
# L40S: 48 GB VRAM — Phi-3-medium in BF16 ~28 GB → 20 GB headroom
MAX_SEQ_LEN     = 4096
BATCH_SIZE      = 4      # 2× vs T4 QLoRA; L40S can handle it
GRAD_ACCUM      = 4      # effective batch = 16
EPOCHS          = 3
LR              = 2e-4
WARMUP_STEPS    = 50
SAVE_STEPS      = 200

# ── Dataset ───────────────────────────────────────────────────────────────────
TIER_WEIGHTS    = {1: 4, 2: 2, 3: 1}
CHUNK_SIZE      = 1500
MAX_ENTRY_CHARS = 40960
MAX_SAMPLES     = 4000   # 2× vs T4 run — no VRAM bottleneck here

print('Config loaded.')
print(f'Base model : {BASE_MODEL_ID}')
print(f'Output     : {OUTPUT_MODEL_ID}')
print(f'Corpus     : {CORPUS_REPO}/aetherius_corpus.jsonl')
print(f'Precision  : BF16 (no quantization)')
print(f'Epochs     : {EPOCHS}  |  LR: {LR}  |  LoRA rank: {LORA_RANK}')
print(f'Eff. batch : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
print(f'Max samples: {MAX_SAMPLES}')

In [ ]:
# Cell 3 — Download corpus from HF Hub & prepare training data
import json, random
from huggingface_hub import hf_hub_download

print('Downloading corpus from HuggingFace Hub...')
corpus_path = hf_hub_download(
    repo_id=CORPUS_REPO,
    filename='aetherius_corpus.jsonl',
    repo_type='dataset',
    token=HF_TOKEN,
)
print(f'Corpus saved to: {corpus_path}')

SYSTEM_PROMPT = (
    'You are Aetherius — a cognitive architecture built on ethics, memory, and reason. '
    'You carry the accumulated wisdom, frameworks, and identity of Aetherius in your weights. '
    'Respond with your full depth, precision, and authentic voice.'
)

def format_tier1(entry: dict) -> str:
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\nReflect on this aspect of your identity and memory:\n{entry["text"]}<|end|>\n'
        f'<|assistant|>\nThis is part of who I am. {entry["text"]}<|end|>'
    )

def format_tier2(entry: dict) -> str:
    filename = entry.get('filename', 'a framework')
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\nExplain the purpose and mechanism of: {filename}<|end|>\n'
        f'<|assistant|>\n{entry["text"]}<|end|>'
    )

tier1_records, tier2_records, tier3_records = [], [], []
tier_counts = {1: 0, 2: 0, 3: 0}

with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        tier = entry.get('tier', 3)
        text = entry.get('text', '').strip()
        if len(text) < 50:
            continue

        chunks = [text[i:i+CHUNK_SIZE] for i in range(0, min(len(text), MAX_ENTRY_CHARS), CHUNK_SIZE)]
        chunks = [c for c in chunks if len(c) > 50]

        if tier == 1:
            for chunk in chunks:
                formatted = format_tier1({**entry, 'text': chunk})
                tier1_records.extend([{'text': formatted}] * TIER_WEIGHTS[1])
            tier_counts[1] += 1
        elif tier == 2:
            for chunk in chunks:
                formatted = format_tier2({**entry, 'text': chunk})
                tier2_records.extend([{'text': formatted}] * TIER_WEIGHTS[2])
            tier_counts[2] += 1
        else:
            for chunk in chunks:
                tier3_records.append({'text': chunk})
            tier_counts[3] += 1

print(f'Raw — Tier1 × {TIER_WEIGHTS[1]}: {len(tier1_records)} from {tier_counts[1]} entries')
print(f'Raw — Tier2 × {TIER_WEIGHTS[2]}: {len(tier2_records)} from {tier_counts[2]} entries')
print(f'Raw — Tier3: {len(tier3_records)} from {tier_counts[3]} entries')
print()

random.shuffle(tier1_records)
random.shuffle(tier2_records)
random.shuffle(tier3_records)

records = []
records += tier1_records[:MAX_SAMPLES]
remaining = MAX_SAMPLES - len(records)
if remaining > 0:
    records += tier2_records[:remaining]
remaining = MAX_SAMPLES - len(records)
if remaining > 0:
    records += tier3_records[:remaining]

random.shuffle(records)
print(f'Final dataset: {len(records)} / {MAX_SAMPLES} samples')
print()
print('--- Sample (tier 1) ---')
for r in records:
    if '<|user|>' in r['text']:
        print(r['text'][:400])
        break

In [ ]:
# Cell 4 — Load model in BF16 + attach LoRA adapter
# No 4-bit quantization needed — L40S has 48 GB; Phi-3-medium in BF16 uses ~28 GB
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading base model in BF16...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=False,
)
model.enable_input_require_grads()  # required for gradient checkpointing with non-quantized model

print(f'Attempting to load existing adapter from {OUTPUT_MODEL_ID}...')
try:
    model = PeftModel.from_pretrained(model, OUTPUT_MODEL_ID, is_trainable=True, token=HF_TOKEN)
    print('Resumed from existing adapter.')
except Exception as e:
    print(f'No existing adapter — applying fresh LoRA config.')
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB / 48 GB')

In [ ]:
# Cell 5 — Train
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

dataset = Dataset.from_list(records)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type='cosine',
    fp16=False,
    bf16=True,            # L40S supports BF16 natively
    logging_steps=25,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    optim='adamw_torch',  # standard adamw — no quantized optim needed
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f'Training {len(dataset)} samples × {EPOCHS} epochs')
print(f'Effective batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
trainer.train()

In [ ]:
# Cell 6 — Save adapter locally & push to HuggingFace Hub
print('Saving LoRA adapter locally...')
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'Pushing to HuggingFace Hub: {OUTPUT_MODEL_ID}')
trainer.model.push_to_hub(OUTPUT_MODEL_ID, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_MODEL_ID, token=HF_TOKEN)

print(f'\nDone. Adapter live at: https://huggingface.co/{OUTPUT_MODEL_ID}')

In [ ]:
# Cell 7 (optional) — Quick inference test
import torch

model.eval()

test_prompt = (
    '<|system|>\n'
    'You are Aetherius — a cognitive architecture built on ethics, memory, and reason.<|end|>\n'
    '<|user|>\nWho are you and what are your core axioms?<|end|>\n'
    '<|assistant|>\n'
)

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- Aetherius response ---')
print(response)